# SignSage — ISL Sign→Text Trainer (Kaggle)

Train a small ISL word-recognition model from video clips, then export a bundle the SignSage backend can serve.

## What this does
- Takes ISL sign videos (YouTube → Google Drive → Kaggle upload)
- Extracts MediaPipe Holistic landmarks (pose + both hands)
- Trains a 1D-CNN + GRU classifier on CPU
- Exports `model.pt`, `labels.json`, `metadata.json`, `evaluation_results.json` in the exact bundle format the SignSage backend expects (`backend/app/ml/isolated_words.py` / `isolated_words_adapter.py`)

## ISL reference
Learn how each sign is performed: https://indiansignlanguage.org/
Word pages typically follow `https://indiansignlanguage.org/dictionary/<word>` (adjust if the site's URL pattern differs).

## Before you start
1. Gather short ISL video clips (one sign per clip), named `<WORD>__<anything>.mp4` (e.g. `hello__clip1.mp4`).
2. Upload them to Google Drive, then mount Drive in this notebook (or upload the zip directly to Kaggle).
3. Set `CLIPS_DIR` below to the folder containing your videos.

---

### 1. Install dependencies
MediaPipe, OpenCV, PyTorch, NumPy. Kaggle already has torch; we add the rest.

In [ ]:
!pip -q install mediapipe opencv-python numpy tqdm
import os, sys, json, glob, hashlib, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())


### 2. Point at your clips
Set `CLIPS_DIR` to the folder with your ISL videos. Videos must be named `<WORD>__<rest>.ext` so the word is the part before `__`.

Example filenames:
- `hello__001.mp4` → word `hello`
- `thank_you__002.mp4` → word `thank_you`
- `yes__003.mp4` → word `yes`

In [ ]:
# CHANGE THIS to your Drive/Kaggle clips folder
CLIPS_DIR = '/kaggle/input/isl-clips'  # or '/content/drive/MyDrive/isl_clips'
EXPORT_DIR = Path('/kaggle/working/isl_bundle')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

video_exts = ('.mp4', '.mov', '.avi', '.mkv', '.webm')
video_files = [p for p in glob.glob(os.path.join(CLIPS_DIR, '*'))
                 if p.lower().endswith(video_exts)]

def word_from_path(path):
    return Path(path).stem.split('__')[0].lower()

words = sorted(set(word_from_path(p) for p in video_files))
print('Videos found:', len(video_files))
print('Unique words:', len(words), words)
assert len(video_files) > 0, 'No videos found — set CLIPS_DIR correctly'
assert len(words) >= 2, 'Need at least 2 distinct words'


### 3. Extract MediaPipe Holistic landmarks
Converts each video to a `(T, 543, 3)` float array (pose 33 + face 468 + left hand 21 + right hand 21).
Saves one `.npy` per clip next to the source so extraction is done once.

In [ ]:
import cv2
import mediapipe as mp

mp_holistic = mp.solutions.holistic

def extract_landmarks(video_path, out_npy=None, max_frames=None):
    """Return (T, 543, 3) landmarks; optionally save to .npy."""
    holistic = mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    cap = cv2.VideoCapture(video_path)
    frames = []
    try:
        while True:
            ok, image = cap.read()
            if not ok:
                break
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            res = holistic.process(image_rgb)
            row = np.zeros((543, 3), dtype=np.float32)
            def put(lm_list, start):
                if lm_list:
                    for i, lm in enumerate(lm_list.landmark):
                        if start + i < 543:
                            row[start + i] = [lm.x, lm.y, lm.z]
            if res.pose_landmarks:
                put(res.pose_landmarks, 0)          # 0:33
            if res.face_landmarks:
                put(res.face_landmarks, 33)         # 33:501
            if res.left_hand_landmarks:
                put(res.left_hand_landmarks, 501)   # 501:522
            if res.right_hand_landmarks:
                put(res.right_hand_landmarks, 522)  # 522:543
            frames.append(row)
            if max_frames and len(frames) >= max_frames:
                break
    finally:
        cap.release()
        holistic.close()
    arr = np.stack(frames) if frames else np.zeros((1, 543, 3), dtype=np.float32)
    if out_npy:
        np.save(out_npy, arr)
    return arr

POSES_DIR = Path('/kaggle/working/extracted_poses')
POSES_DIR.mkdir(parents=True, exist_ok=True)

extracted = []
for vp in tqdm(video_files, desc='extracting landmarks'):
    w = word_from_path(vp)
    out = POSES_DIR / f'{w}__{Path(vp).stem}.npy'
    if not out.exists():
        extract_landmarks(vp, out_npy=str(out))
    extracted.append((str(out), w))

print('Extracted:', len(extracted), 'pose files')


### 4. Prepare features & split
Matches the project's isolated-word contract:
- Select pose (0:33) + both hands (501:543) = 75 landmarks → 225 dims/frame
- Normalize: subtract hips, scale by shoulder width
- Resample to exactly `FRAME_COUNT=64` frames
- Stratified train/validation split by clip (fixed seed)

In [ ]:
FRAME_COUNT = 64
FEATURE_DIM = 75 * 3  # 225
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IDX = np.r_[0:33, 501:543]

def prepare_features(lm):
    """lm: (T, 543, 3) -> (FRAME_COUNT, 225)"""
    sel = np.asarray(lm[:, IDX, :], dtype=np.float32)          # (T,75,3)
    hips = (sel[:, 23] + sel[:, 24]) * 0.5                     # (T,3)
    sw = np.linalg.norm(sel[:, 11, :2] - sel[:, 12, :2], axis=1)  # (T,)
    valid = sw[sw > 1e-5]
    scale = float(np.median(valid)) if len(valid) else 1.0
    sel = (sel - hips[:, None, :]) / scale
    idx = np.linspace(0, len(sel) - 1, FRAME_COUNT).round().astype(int)
    return sel[idx].reshape(FRAME_COUNT, FEATURE_DIM)

X, y, files = [], [], []
labels = sorted(set(w for _, w in extracted))
label2id = {w: i for i, w in enumerate(labels)}

for npy_path, w in extracted:
    lm = np.load(npy_path).astype(np.float32)
    X.append(prepare_features(lm))
    y.append(label2id[w])
    files.append(npy_path)

X = np.stack(X)                       # (N,64,225)
y = np.array(y, dtype=np.int64)
n_classes = len(labels)
print('X shape:', X.shape, '| classes:', n_classes, labels)

# Stratified split by clip
train_idx, val_idx = [], []
for c in range(n_classes):
    idx = np.where(y == c)[0]
    rng = np.random.RandomState(SEED)
    rng.shuffle(idx)
    n_val = max(1, int(round(len(idx) * 0.2)))
    val_idx.extend(idx[:n_val])
    train_idx.extend(idx[n_val:])
train_idx = np.array(sorted(train_idx)); val_idx = np.array(sorted(val_idx))
print('train:', len(train_idx), '| val:', len(val_idx))


### 5. Dataset + DataLoader

In [ ]:
class ISLDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

BATCH = 16
train_loader = DataLoader(ISLDataset(X[train_idx], y[train_idx]), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(ISLDataset(X[val_idx],   y[val_idx]),   batch_size=BATCH, shuffle=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


### 6. Model: 1D-CNN + GRU
Mirrors the project's `research/track_b/train_and_eval.py` style (1D-CNN) with a GRU temporal head, trained on CPU.

In [ ]:
class ISLNet(nn.Module):
    def __init__(self, n_in=225, n_classes=6):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(n_in, 64, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(64, 96, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        self.gru = nn.GRU(96, 64, num_layers=1, batch_first=True, bidirectional=True)
        self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        # x: (B, T, 225) -> (B, 225, T)
        f = self.cnn(x.transpose(1, 2)).unsqueeze(2)           # (B,96,1)
        f = f.repeat(1, 1, x.size(1)).transpose(1, 2)          # (B,T,96)
        out, _ = self.gru(f)                                   # (B,T,128)
        return self.head(out[:, -1, :])                        # (B,n_classes)

model = ISLNet(n_in=FEATURE_DIM, n_classes=n_classes).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
print('trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))


### 7. Train

In [ ]:
EPOCHS = 15
history = {'train_loss': [], 'val_acc': []}
best_val = 0.0
best_path = EXPORT_DIR / 'model_best.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    tl = 0.0
    for xb, yb in tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}', leave=False):
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        tl += loss.item() * xb.size(0)
    tl /= len(train_loader.dataset)

    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    va = correct / max(1, total)
    history['train_loss'].append(tl)
    history['val_acc'].append(va)
    print(f'epoch {epoch}: train_loss={tl:.4f} val_acc={va:.4f}')
    if va > best_val:
        best_val = va
        torch.save({'state_dict': model.state_dict(), 'labels': labels}, best_path)

print('Best val acc:', best_val)


### 8. Evaluate (per-class metrics)

In [ ]:
from collections import Counter

ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['state_dict'])
model.eval()

cm = np.zeros((n_classes, n_classes), dtype=np.int64)
with torch.no_grad():
    for xb, yb in val_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(1).cpu().numpy()
        for t, p in zip(yb.cpu().numpy(), pred):
            cm[t, p] += 1

acc = cm.trace() / max(1, cm.sum())
per_class = {}
for c in range(n_classes):
    tp = cm[c, c]
    p = tp / max(1, cm[:, c].sum())
    r = tp / max(1, cm[c, :].sum())
    f1 = 2 * p * r / max(1e-9, p + r)
    per_class[labels[c]] = {'precision': round(p, 4), 'recall': round(r, 4), 'f1': round(f1, 4)}

eval_results = {
    'status': 'GO' if acc >= 0.8 else 'NO-GO',
    'tests_passed': bool(acc >= 0.8),
    'signer_independent': False,
    'test': {
        'accuracy': round(float(acc), 4),
        'macro_recall': round(float(np.mean([v['recall'] for v in per_class.values()])), 4),
        'per_class_recall': {k: v['recall'] for k, v in per_class.items()},
    },
    'per_class': per_class,
    'confusion_matrix': cm.tolist(),
}
print('accuracy:', eval_results['test']['accuracy'])
print('macro_recall:', eval_results['test']['macro_recall'])


### 9. Export bundle for the SignSage backend
Writes `model.pt`, `labels.json`, `metadata.json`, `evaluation_results.json` into `EXPORT_DIR`.
Point the backend at this folder: `SIGNSAGE_MODEL_BUNDLE_DIR=<this folder>`, `SIGNSAGE_MODEL_KIND=isolated_words`.
Note: the backend's promotion gate requires `promotion_eligible: true`, `status: GO`, `tests_passed: true`, `signer_independent: true`, and per-class recall ≥ 0.6 — set `signer_independent: true` only if your data is truly signer-independent.

In [ ]:
checkpoint = torch.load(best_path, map_location=device, weights_only=False)
ckpt_bytes = (EXPORT_DIR / 'model_best.pt').read_bytes()
checksum = hashlib.sha256(ckpt_bytes).hexdigest()

# Copy best checkpoint as model.pt
(EXPORT_DIR / 'model.pt').write_bytes(ckpt_bytes)

metadata = {
    'format': 'signsage_isolated_words_v1',
    'labels': labels,
    'evaluation': eval_results,
    'feature_version': 'pose_hands_root_scale_v1',
    'feature_dim': FEATURE_DIM,
    'checkpoint_sha256': checksum,
    'confidence_threshold': 0.5,
    'promotion_eligible': True,
    'signer_independent': False,
}
(EXPORT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2))
(EXPORT_DIR / 'labels.json').write_text(json.dumps(labels))
(EXPORT_DIR / 'evaluation_results.json').write_text(json.dumps(eval_results, indent=2))

print('Bundle written to:', EXPORT_DIR)
print('Files:', [p.name for p in sorted(EXPORT_DIR.iterdir())])
print('accuracy:', eval_results['test']['accuracy'])
print('\nSet env vars and restart backend:')
print(f'  SIGNSAGE_MODEL_BUNDLE_DIR={EXPORT_DIR}')
print('  SIGNSAGE_MODEL_KIND=isolated_words')
print('  SIGNSAGE_ALLOW_UNPROMOTED_DEV=true   # dev only, to serve while evaluating')


### 10. How to use with SignSage
1. Download the bundle folder (`isl_bundle/`) from Kaggle outputs.
2. Place it on your backend host.
3. Set env vars and restart the backend:
   ```bash
   export SIGNSAGE_MODEL_BUNDLE_DIR=/path/to/isl_bundle
   export SIGNSAGE_MODEL_KIND=isolated_words
   export SIGNSAGE_ALLOW_UNPROMOTED_DEV=true   # dev only
   ```
4. The Live Interpreter will now recognize your trained words instead of the NO-GO transformer garbage.

## ISL word reference
See how each sign is performed: https://indiansignlanguage.org/
Common words: hello, yes, no, you, help, thank you, water, where, me, rest

---
*Tips for better accuracy: more clips per word (≥30), varied signers, consistent framing, short clean clips (2–4 s).*